# 01 · Khám phá dữ liệu (Data Exploration)

## Mục tiêu
Hiểu **cấu trúc** và **chất lượng** của dataset Parkinson trước khi tiền xử lý / mô hình hoá:
kích thước, tên cột, kiểu dữ liệu (dtype), cột nhãn, số mẫu theo nhãn, thống kê mô tả (min/max/mean),
và kiểm tra dữ liệu thiếu / trùng / bất thường.

## Yêu cầu bài tập liên quan
- Xác minh cấu trúc & dtype (M02-T1), phân nhóm đặc trưng (M02-T2), phát hiện vấn đề (M02-T3),
  thống kê mô tả (M03-T1), kiểm tra chất lượng (M03-T2), báo cáo chất lượng (M03-T3).

## Phạm vi của notebook này
- **M02** (khám phá cấu trúc/dtype) + **M03** (thống kê mô tả + chất lượng).

## KHÔNG thuộc phạm vi
- Tiền xử lý / PCA (→ notebook 02), phân lớp (→ 03), phân cụm (→ 04), tổng hợp (→ 05).


## Phân công (theo `TASK_ASSIGNMENTS.md`)

| Nhiệm vụ | Phụ trách | Reviewer |
|---|---|---|
| M02-T1 Khám phá cấu trúc & dtype | M02 | M03 |
| M02-T2 Bản đồ nhóm đặc trưng | M02 | M03 |
| M02-T3 Phát hiện vấn đề (scale/outlier) | M02 | M04 |
| M03-T1 Thống kê mô tả | M03 | M02 |
| M03-T2 Kiểm tra chất lượng | M03 | M04 |
| M03-T3 Báo cáo chất lượng (gate M01) | M03 | M01 |

**Tên / MSSV thành viên:** M02 = **Lê Thiên Lộc** (MSSV `079306040024`) · M03 = **Huỳnh Gia Hợp** (MSSV `054205008367`) (xem `TEAM.md`).


## Điều kiện bắt đầu (đọc kỹ trước khi làm)

- **Đầu vào:** canonical CSV `data/processed/parkinsons_v1.csv` (do **M01-T2** tạo, xác nhận bởi **S-T1**).
  Dữ liệu chuẩn hoá: **22 feature + cột `status`**, đã loại cột `name` (xem `DATA_DICTIONARY.md`).
- **Nếu CSV chuẩn CHƯA tồn tại** → **DỪNG**, đánh dấu **`Blocked`** trong `TASK_BOARD.md`
  (chờ M01-T2 / S-T1). **KHÔNG tự tạo CSV** ở notebook này (vi phạm quy ước `data/README.md`).
- **Điều kiện chạy:** chạy từ **gốc project** (đường dẫn tương đối), không hardcode đường dẫn máy cá nhân.


In [ ]:
import os

CSV = "data/processed/parkinsons_v1.csv"
if not os.path.exists(CSV):
    raise FileNotFoundError(
        "BLOCKED: chưa có canonical CSV -> chờ M01-T2 (S-T1); "
        "không tự tạo CSV trong notebook này."
    )
print(f"Canonical CSV: {CSV}")

## Bước 1 · Kiểm tra nguồn dữ liệu
**Mục tiêu:** xác nhận đúng file / đường dẫn và file đọc được.
**Vì sao:** sai nguồn → mọi thống kê sau vô nghĩa. Canonical CSV là đầu vào DUY NHẤT.


In [ ]:
# M02-T1: đọc đúng canonical CSV, không đọc/tạo bản CSV riêng
import pandas as pd

df = pd.read_csv(CSV)
df.head()

## Bước 2 · Kích thước dữ liệu
**Mục tiêu:** biết số dòng (mẫu) và số cột (thuộc tính) thực tế.
**Vì sao:** đối chiếu với số đã xác minh (canonical: 22 feature + `status` = 23 cột, ~195 dòng).
**Câu hỏi tự kiểm:** số dòng/cột có khớp kỳ vọng không? Nếu lệch → ghi nhận và tra nguyên nhân.


In [ ]:
# M02-T1: kích thước thực tế
print(f"Số dòng, số cột: {df.shape}")
assert df.shape == (195, 23), f"Kích thước khác kỳ vọng: {df.shape}"

## Bước 3 · Tên cột & kiểu dữ liệu
**Mục tiêu:** liệt kê tên cột và dtype từng cột.
**Vì sao:** nhóm feature liên tục (FREQ/JITTER/SHIMMER/NOISE/NONLIN) sẽ cần chuẩn hoá sau (notebook 02);
cột `status` phải là số nhị phân (0/1).


In [ ]:
# M02-T1: tên cột, dtype và kiểm tra kiểu dữ liệu
print("Các cột:", list(df.columns))
print(df.dtypes)
features = [column for column in df.columns if column != "status"]
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in features)
assert pd.api.types.is_integer_dtype(df["status"])
assert set(df["status"].unique()) == {0, 1}
print("Đã xác nhận 22 feature là số và status là nhãn nguyên nhị phân 0/1.")

## Bước 4 · Xác định thuộc tính & cột nhãn (theo tài liệu ĐÃ XÁC MINH)
**Mục tiêu:** phân biệt **feature (22 cột)** vs **label `status`**; xác nhận **`name` đã được loại**.
**Nguồn:** `DATA_DICTIONARY.md` (đã xác minh từ dataset).
- **Nhóm feature:** FREQ (3) · JITTER (5) · SHIMMER (6) · NOISE: `NHR`,`HNR` · NONLIN: `RPDE`,`DFA`,`spread1`,`spread2`,`D2`,`PPE` (6).
- **Label:** `status` (`0`=khỏe, `1`=Parkinson).
**Câu hỏi tự kiểm:** canonical CSV có còn cột `name` không? (Phải **KHÔNG**.) Có đủ 22 feature + `status` không?


In [ ]:
# M02-T1/T2: xác nhận schema và bản đồ nhóm đặc trưng
assert "name" not in df.columns, "LỖI: canonical CSV vẫn giữ cột 'name'"
assert "status" in df.columns, "LỖI: thiếu cột nhãn 'status'"
assert len(features) == 22, f"Số feature phải là 22, nhận được {len(features)}"

feature_groups = {
    "FREQ": ["MDVP:Fo(Hz)", "MDVP:Fhi(Hz)", "MDVP:Flo(Hz)"],
    "JITTER": ["MDVP:Jitter(%)", "MDVP:Jitter(Abs)", "MDVP:RAP", "MDVP:PPQ", "Jitter:DDP"],
    "SHIMMER": ["MDVP:Shimmer", "MDVP:Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5", "MDVP:APQ", "Shimmer:DDA"],
    "NOISE": ["NHR", "HNR"],
    "NONLIN": ["RPDE", "DFA", "spread1", "spread2", "D2", "PPE"],
}
assert sorted(sum(feature_groups.values(), [])) == sorted(features)
for group, columns in feature_groups.items():
    print(f"{group} ({len(columns)}): {', '.join(columns)}")

## Bước 5 · Số mẫu theo từng nhãn
**Mục tiêu:** đếm số mẫu ở mỗi giá trị `status`.
**Vì sao:** biết lớp có **cân bằng** không → ảnh hưởng cách chia train/test & chọn metric (notebook 03).
**Câu hỏi tự kiểm:** tỉ lệ 0/1 là bao nhiêu? Mất cân bằng có đáng kể không?


In [ ]:
# TODO (M03-T1): đếm mẫu theo nhãn
# df["status"].value_counts()
# df["status"].value_counts(normalize=True)   # tỉ lệ theo lớp
# Sinh viên: ghi nhận số mẫu lớp 0 và lớp 1 (đừng điền sẵn ở đây)
pass

## Bước 6 · min / max / mean cho thuộc tính số
**Mục tiêu:** thống kê mô tả các feature liên tục.
**Vì sao:** phát hiện chênh lệch scale (vd `MDVP:Jitter(Abs)` rất nhỏ, `MDVP:Shimmer` khác)
→ cơ sở để chuẩn hoá (notebook 02).
**Làm gì:** dùng `describe()` / `min()`,`max()`,`mean()`; so sánh **giữa các nhóm** (FREQ vs JITTER vs NONLIN).


In [ ]:
# TODO (M03-T1): thống kê min/max/mean theo từng feature
# df.describe().T        # mỗi feature một hàng: count/mean/min/25%/50%/75%/max
# Sinh viên: nhận xét feature nào có scale rất nhỏ / rất lớn (dựa trên KẾT QUẢ THỰC TẾ)
pass

## Bước 7 · Dữ liệu thiếu, trùng lặp, bất thường
**Mục tiêu:** kiểm tra NaN, hàng trùng, giá trị ngoài khoảng (candidate outlier).
**Vì sao:** dữ liệu bẩn phải được xử lý TRƯỚC khi mô hình (đề xuất phương pháp cho M04).
**Câu hỏi tự kiểm:** có NaN không? Có hàng trùng không? Có giá trị rõ ràng bất thường không?


In [ ]:
# M02-T1/T3: thiếu, trùng và candidate outlier theo quy tắc IQR.
missing_by_column = df.isna().sum()
duplicate_rows = int(df.duplicated().sum())
numeric_summary = df[features].describe().T
q1 = df[features].quantile(0.25)
q3 = df[features].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df[features] < q1 - 1.5 * iqr) | (df[features] > q3 + 1.5 * iqr)
outlier_counts = outlier_mask.sum().sort_values(ascending=False)

print("Tổng missing:", int(missing_by_column.sum()))
print("Hàng trùng:", duplicate_rows)
print("Candidate outlier theo feature:")
print(outlier_counts[outlier_counts > 0])
print("Các feature có scale/range chênh lệch lớn cần chuẩn hoá trước PCA/mô hình:")
print((numeric_summary["max"] - numeric_summary["min"]).sort_values())

## Kết quả & nhận xét M02 (đã chạy trên canonical CSV v1)

| Hạng mục | Giá trị đo được | Nhận xét |
|---|---:|---|
| Số dòng (mẫu) | 195 | Khớp schema v1 |
| Số cột (canonical) | 23 | 22 feature + `status`, không có `name` |
| Cột nhãn | `status` | Giá trị hợp lệ 0/1 |
| Số mẫu label=0 / label=1 | 48 / 147 | Hai lớp lệch nhau; M03 phụ trách đánh giá chất lượng |
| NaN (tổng) | 0 | Chưa cần xử lý thiếu ở bước khám phá |
| Hàng trùng | 0 | Không phát hiện trùng toàn hàng |
| Feature có scale chênh lệch | Có | FREQ có range hàng trăm, JITTER có giá trị rất nhỏ; cần chuẩn hoá trước PCA/mô hình |
| Candidate outlier | Có ở nhiều feature | Dùng IQR để đánh dấu ứng viên, không tự xoá; M03/M04 quyết định xử lý |

**Bàn giao M02:**
- M03: cấu trúc đã xác nhận; lưu ý phân bố nhãn 48/147 và các candidate outlier theo bảng chạy trong cell Bước 7.
- M04: cần chuẩn hoá vì scale giữa FREQ/JITTER/SHIMMER/NOISE/NONLIN khác biệt; không loại outlier tự động khi chưa có quy ước.


## Checklist M02

- [x] Đã chạy guard và đọc canonical CSV v1 (không CSV riêng).
- [x] Đã xác minh kích thước, tên cột, dtype.
- [x] Đã xác nhận `name` đã drop, `status` là nhãn, đủ 22 feature.
- [x] Đã kiểm tra missing, trùng và candidate outlier.
- [x] Đã ghi nhận chênh lệch scale và nhu cầu chuẩn hoá cho M04.
- [x] Đã cập nhật `DATA_DICTIONARY.md` và `results/m02_exploration.md`.
- [x] Đã chuyển M02-T1..T3 sang `Review` trong `TASK_BOARD.md`.

### Phạm vi M03 còn lại
- Bước 5–6 (thống kê phân bố nhãn và thống kê mô tả chi tiết) vẫn do M03 phụ trách.
- M03-T2/T3 quyết định xử lý chất lượng dựa trên kết quả M02 và kiểm tra riêng.


## Tái lập & xử lý Blocked

- **Nguồn dữ liệu:** canonical CSV `data/processed/parkinsons_v1.csv` (version v1, do M01-T2 tạo).
- **Đầu vào:** canonical CSV + `DATA_DICTIONARY.md`. **Đầu ra dự kiến:** bảng khám phá + `results/quality_report.md`.
- **Không hardcode** đường dẫn máy cá nhân — dùng đường dẫn tương đối (chạy từ gốc project).
- **Ghi lại** (vào `results/`): seed nếu dùng, phiên bản thư viện, cách xử lý NaN/trùng (nếu chọn).
- **Nếu thiếu đầu vào / phụ thuộc chưa xong:** đánh dấu **`Blocked`** trong `TASK_BOARD.md` + ghi rõ
  thiếu gì, ai cung cấp; thông báo M01 (điều phối). Xem `HANDOFF.md` mục 6.
